In [ ]:
from pathlib import Path
import ixmp4
import pyam
import nomenclature

In [ ]:
df = pyam.concat(
    [
        i for i in list(Path("raw/ENGAGE_T34/").iterdir())
    ]
)

In [ ]:
# rename scenarios for clear reference to ENGAGE
df.rename(
    scenario=dict(
        [
            (
                i,
                "ENGAGE-Feasibility-"
                + (
                    i[4:]               
                    .replace("1000_", "1000/")
                    .replace("m550_", "Maximum-Effort/")
                    .replace("bitb_ref", "Technology")
                    .replace("ref", "Cost-Effective")
                    .replace("govem", "Institutional")
                    .replace("govpr", "Institutional [Optimistic]")
                    .replace("bitb_em", "Technology & Institutional")
                    .replace("bitb_pr", "Technology & Institutional [Optimistic]")
                    .replace("enab_em", "Enablers & Institutional")
                    .replace("enab_pr", "Enablers & Institutional [Optimistic]")
                    .replace("feas_em", "Technology & Enablers & Institutional")
                    .replace("feas_pr", "Technology & Enablers & Institutional [Optimistic]")
                    .replace("_f", " [Pessimistic]")
                )
            ) for i in df.scenario
        ]
    ),
    inplace=True,
)

In [ ]:
df.scenario

In [ ]:
#definition = nomenclature.DataStructureDefinition("../definitions/")
definition = nomenclature.DataStructureDefinition("../../common-definitions/definitions/")

In [ ]:
validation_args = ["upper_bound", "lower_bound", "value", "rtol", "atol", "range"]

validation_list = list()

for name, variable in definition.variable.items():
    if any([i in validation_args for i in variable.extra_attributes]):
        validation_list.append(
            dict(
                variable=name,
                validation=[dict([(key, value) for key, value in variable.extra_attributes.items() if key in validation_args])]
            )
        )

In [ ]:
validator = nomenclature.processor.DataValidator(criteria_items=validation_list, file=".")

In [ ]:
validator

In [ ]:
validator.apply(df)

In [ ]:
# remove variables of little relevance that are not included in common-definitions
df.filter(
    variable=[
        "*AR5 climate diagnostics*",
        "Diagnostics|MAGICC6*",
        "Carbon Sequestration|Other",
        "Secondary Energy",
        "Food Energy Supply",
        "Investment|Energy Supply|Electricity|Non-fossil",
        "Investment|Energy Supply|Extraction|Bioenergy",
        "Investment|Energy Supply|Hydrogen|Renewable",
        "Policy Cost|Consumption Loss",
        "Policy Cost|GDP Loss",
        "Policy Cost|Area under MAC Curve",
        "Price|Agriculture|Non-Energy Crops and Livestock|Index",
        "Policy Cost|Additional Total Energy System Cost",
        "Carbon Sequestration|CCS|Biomass|Energy|Demand|Industry", 
        "Final Energy|Residential and Commercial|Solids|Biomass|Traditional",
        "Final Energy|Transportation|Liquids|Natural Gas",
        "Capacity Additions|Electricity|Storage Capacity",
        "Capacity|Electricity|Peak Demand",
        "Capacity|Electricity|Storage",
        "Secondary Energy|Electricity|Curtailment",
        "Secondary Energy|Electricity|Curtailment|Solar",
        "Secondary Energy|Electricity|Curtailment|Wind",
        "Secondary Energy|Electricity|Storage",
        "Secondary Energy|Electricity|Storage Losses",
        "Secondary Energy|Electricity|Transmission Losses",
    ],
    keep=False,
    inplace=True
)

In [ ]:
# rename units
df.rename(
    unit={
        "US$2010/kW OR local currency/kW": "USD_2010/kW",
        "US$2010/kW": "USD_2010/kW",
        "billion US$2010/yr": "billion USD_2010/yr",
        "billion US$2010/yr OR local currency/yr": "billion USD_2010/yr",
        "billion US$2010/yr or local currency/yr": "billion USD_2010/yr",
        "US$2010/t CO2": "USD_2010/t CO2",
        "US$2010/tCO2": "USD_2010/t CO2",
        "US$2010/t CO2 or local currency/t CO2": "USD_2010/t CO2",
        "million Ha/yr": "million ha",
        "Million": "million",
        "Mt NOx/yr": "Mt NO2/yr",  
        "Mt N2O/yr": "kt N2O/yr",
        "million m3/yr": "km3/yr",
    },
    inplace=True,
)

In [ ]:
# update carbon-management variables
carbon_management_mapping = {
    "Carbon Sequestration|CCS": "Carbon Capture|Geological Storage",
    "Carbon Sequestration|CCS|Biomass": "Carbon Capture|Geological Storage|Biomass",
    "Carbon Sequestration|CCS|Biomass|Energy|Supply": "Carbon Capture|Energy|Supply|Biomass",
    "Carbon Sequestration|CCS|Fossil": "Carbon Capture|Energy|Fossil",
    "Carbon Sequestration|CCS|Fossil|Energy|Demand|Industry": "Carbon Capture|Energy|Demand|Industry",
    "Carbon Sequestration|CCS|Fossil|Energy|Supply": "Carbon Capture|Energy|Supply|Fossil",
    "Carbon Sequestration|CCS|Industrial Processes": "Carbon Capture|Industrial Processes",
    "Carbon Sequestration|Land Use|Afforestation": "Carbon Removal|Land Use|Re/Afforestation",
    "Carbon Sequestration|Direct Air Capture": "Carbon Removal|Geological Storage|Direct Air Capture",
    "Carbon Sequestration|Enhanced Weathering": "Carbon Removal|Enhanced Weathering",
    "Carbon Sequestration|Land Use": "Carbon Removal|Land Use",
}

df.rename(variable=carbon_management_mapping, inplace=True)

In [ ]:
project = "engage"
legacy_mapping = {}

for code, attrs in definition.variable.items():
    if project in attrs.extra_attributes:
        legacy_mapping[attrs.__getattr__(project)] = code

df.rename(variable=legacy_mapping, inplace=True)

In [ ]:
df.filter(variable="Capacity Additions|Electricity|*", unit="GW", keep=False, inplace=True)

In [ ]:
df.model

In [ ]:
df.rename(
    model={
        "GEM-E3_V2023": "GEM-E3 V2023",
        "MESSAGEix-GLOBIOM_1.1": "MESSAGEix-GLOBIOM 1.1",
        "POLES ENGAGE": "POLES-JRC ENGAGE",
    },
    inplace=True,
)

In [ ]:
df.rename(
    region=dict(
        [(i, i.replace("MESSAGEix-GLOBIOM_1.1", "MESSAGEix-GLOBIOM 1.1")) for i in df.filter(region="MESSAGEix*").region]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    region=dict(
        [(i, i.replace("GEM-E3_V2023", "GEM-E3 V2023")) for i in df.filter(region="GEM-E3*").region]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    region={
        "MESSAGEix-GLOBIOM 1.1|Sub-saharan Africa": "MESSAGEix-GLOBIOM 1.1|Sub-Saharan Africa",
        "COFFEE 1.5|European Union": "COFFEE 1.5|Europe",
        "GEM-E3 V2023|Rest of fossil fuel producers": "GEM-E3 V2023|Rest of Fossil Fuel Producers",
        "GEM-E3 V2023|Rest of the world": "GEM-E3 V2023|Rest of the World",
        "IMAGE 3.2|C. Europe": "IMAGE 3.2|Central Europe",
        "IMAGE 3.2|China": "IMAGE 3.2|China Region",
        "IMAGE 3.2|E. Africa": "IMAGE 3.2|Eastern Africa",
        "IMAGE 3.2|Indonesia": "IMAGE 3.2|Indonesia Region",
        "IMAGE 3.2|Kazakhstan region": "IMAGE 3.2|Central Asia",
        "IMAGE 3.2|Korea": "IMAGE 3.2|Korea Region",
        "IMAGE 3.2|N. Africa": "IMAGE 3.2|Northern Africa",
        "IMAGE 3.2|Rest C. America": "IMAGE 3.2|Central America",
        "IMAGE 3.2|Rest S. Africa": "IMAGE 3.2|Rest of Southern Africa",
        "IMAGE 3.2|Rest S. America": "IMAGE 3.2|Rest of South America",
        "IMAGE 3.2|Rest S. Asia": "IMAGE 3.2|Rest of South Asia",
        "IMAGE 3.2|Russia": "IMAGE 3.2|Russia Region",
        "IMAGE 3.2|SE. Asia": "IMAGE 3.2|Southeastern Asia",
        "IMAGE 3.2|USA": "IMAGE 3.2|United States",
        "IMAGE 3.2|Ukraine region": "IMAGE 3.2|Ukraine Region",
        "IMAGE 3.2|W. Africa": "IMAGE 3.2|Western Africa",
        "IMAGE 3.2|W. Europe": "IMAGE 3.2|Western Europe",
        "REMIND 3.0|China": "REMIND 3.0|China and Taiwan",
        "REMIND 3.0|Countries from the Reforming Economies of the Former Soviet Union": "REMIND 3.0|Russia and Reforming Economies",
        "REMIND 3.0|Middle East, North Africa, Central Asia": "REMIND 3.0|Middle East and North Africa",
        "REMIND 3.0|Canada, NZ, Australia": "REMIND 3.0|Canada, Australia, New Zealand",
        "REMIND 3.0|Sub-saharan Africa": "REMIND 3.0|Sub-Saharan Africa",
        "REMIND 3.0|other Asia": "REMIND 3.0|Other Asia",
        "WITCH 5.0|Japan and South Korea": "WITCH 5.0|Japan and Korea",
        "WITCH 5.0|Latin america and Caraibes (except Brasil and Mexico)": "WITCH 5.0|Latin America and the Caribbean",  
        "WITCH 5.0|Moyen-Orient and North Africa": "WITCH 5.0|Middle East and North Africa", 
        "WITCH 5.0|Oceania": "WITCH 5.0|Australia, New Zealand, and Oceania islands", 
        "WITCH 5.0|South-East Asia": "WITCH 5.0|South East Asia", 
        "WITCH 5.0|Sub-Saharian African (except South Africa)": "WITCH 5.0|Sub-Saharan Africa", 
        "WITCH 5.0|Transition Economies (including Russia)": "WITCH 5.0|Non-EU Eastern European and Transition Countries", 
        "WITCH 5.0|United States of America": "WITCH 5.0|United States", 
        "POLES ENGAGE|Africa - Middle East": "Middle East & Africa (R5)",
        "POLES ENGAGE|Asia (excl. Japan)": "Asia (R5)",
        "POLES ENGAGE|Former CIS": "Reforming Economies (R5)",
        "POLES ENGAGE|OECD90+EU": "OECD & EU (R5)",
        "POLES ENGAGE|Latin America": "Latin America (R5)",
        "POLES ENGAGE|Arfica (10R Map)": "Africa (R10)",
        "POLES ENGAGE|China+ (10R Map)": "China+ (R10)",
        "POLES ENGAGE|Europe (10R Map)": "Europe (R10)",
        "POLES ENGAGE|Indai+ (10R Map)": "India+ (R10)",
        "POLES ENGAGE|Latin Amarica (10R Map": "Latin America (R10)",
        "POLES ENGAGE|Middle East (10R Map)": "Middle East (R10)",
        "POLES ENGAGE|North America (10R Map)": "North America (R10)",
        "POLES ENGAGE|Pacific OECD (10R Map)": "Pacific OECD (R10)",
        "POLES ENGAGE|Reforming Econonies (10R Map)": "Reforming Economies (R10)",
        "POLES ENGAGE|Rest of Asia (10R Map)": "Rest of Asia (R10)",
        "POLES ENGAGE|Rest of World (10R Map)": "Other (R10)",
    },
    inplace=True,
)

In [ ]:
definition.validate(df)

In [ ]:
df.set_meta("ENGAGE [Horizon 2020]", "Project")
df.set_meta("Bertram et al. (2024)", "Scientific Manuscript (Citation)")
df.set_meta("10.1038/s41558-024-02073-4", "Scientific Manuscript (DOI)")

In [ ]:
region_processor = nomenclature.RegionProcessor.from_directory("../../common-definitions/mappings/", dsd=definition)

In [ ]:
df_processed = nomenclature.process(region_processor.revert(df), definition, processor=region_processor)

In [ ]:
platform = ixmp4.Platform("scenariocompass-transfer")

In [ ]:
for model in df_processed.model:
    df_processed.filter(model=model).to_ixmp4(platform)
    print(model)